# 02. Phase 2 — Flow Matching 학습
돌연변이 조건부 Transformer velocity network 학습 (T4 기준 4-6시간)

In [ ]:
# 환경 설정
import os, sys
os.chdir('/content')
if not os.path.exists('brain_idp_flow'):
    !git clone https://github.com/layeredlabs06/brain-idp-flow.git brain_idp_flow
os.chdir('brain_idp_flow')
!pip install -e . -q
!pip install fair-esm tensorboard -q
sys.path.insert(0, '/content/brain_idp_flow/src')

import torch
import yaml
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

In [ ]:
# 임베딩이 없으면 01에서 계산 (또는 여기서 직접)
os.makedirs('data', exist_ok=True)

if not os.path.exists('data/seq_embeddings.pt'):
    print('seq_embeddings.pt 없음 — 직접 계산합니다...')
    from brain_idp_flow.targets import load_targets
    from brain_idp_flow.features.seq_embed import ESM2Embedder
    
    targets = load_targets('configs/targets.yaml')
    embedder = ESM2Embedder(device=device)
    seq_embeddings = {}
    sid = 0
    for tid, target in targets.items():
        emb = embedder.embed_single(target.sequence)
        seq_embeddings[sid] = emb.cpu()
        sid += 1
        for mut in target.mutations:
            emb = embedder.embed_single(target.mutant_sequence(mut))
            seq_embeddings[sid] = emb.cpu()
            sid += 1
    torch.save(seq_embeddings, 'data/seq_embeddings.pt')
    print(f'{len(seq_embeddings)} embeddings computed')

# 데이터셋도 없으면 빌드
if not os.path.exists('data/train.npz'):
    !python scripts/build_dataset.py --out data/train.npz --max-len 160

print('Data ready')

In [ ]:
# 학습 실행
from brain_idp_flow.train import train

with open('configs/flow.yaml') as f:
    config = yaml.safe_load(f)

# 캐시에서 임베딩 로드
seq_embeddings = torch.load('data/seq_embeddings.pt', weights_only=True)

# 학습 시작 (T4 기준 ~4-6시간 for 50k steps)
best_ckpt = train(config, seq_embeddings, device)
print(f'\nBest checkpoint: {best_ckpt}')

In [ ]:
# TensorBoard 확인 (선택)
%load_ext tensorboard
%tensorboard --logdir runs/flow

In [ ]:
# 체크포인트를 Google Drive에 백업 (Colab 세션 종료 대비)
from google.colab import drive
drive.mount('/content/drive')

import shutil, glob
ckpt_dir = sorted(glob.glob('runs/flow/*'))[-1]  # latest
backup_dir = '/content/drive/MyDrive/brain_idp_flow_ckpts'
os.makedirs(backup_dir, exist_ok=True)
for f in ['ckpt_best.pt', 'ckpt_last.pt']:
    src = os.path.join(ckpt_dir, f)
    if os.path.exists(src):
        shutil.copy2(src, backup_dir)
        print(f'Backed up {f} -> {backup_dir}')